MDP PRogram
> a robot navigating a 3x3 grid with one obstacle

In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class GridWorldMDP(gym.Env):
    def __init__(self):
        super(GridWorldMDP, self).__init__()

        # grid dimensions
        self.rows = 3
        self.cols = 3

        # define action space: 0=up, 1=down, 2=left, 3=right
        self.action_space = spaces.Discrete(4)

        # define state space: (row, col) positions, excluding (1,1)
        # (1,1) -> obstacle
        self.state_space = [(r, c) for r in range(self.rows) for c in range(self.cols) if (r, c) != (1, 1)]
        self.observation_space = spaces.Discrete(len(self.state_space))

        # map state (r,c)
        self.state_to_idx = {state: idx for idx, state in enumerate(self.state_space)}
        self.idx_to_state = {idx: state for idx, state in enumerate(self.state_space)}
        # obstacle -> (1,1)
        self.wall = (1, 1)
        # Target -> (2,2)
        self.target = (2, 2)
        # start -> (0,0)
        self.start_pos = (0, 0)
        self.current_pos = self.start_pos

        # transition probabilities
        self.p_intended = 0.8
        self.p_perpendicular = 0.1

    # reset function
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_pos = self.start_pos
        state = self.state_to_idx[self.current_pos]
        return state, {}

    # time step function
    def step(self, action):
        current_row, current_col = self.current_pos
        # defining moves
        if action == 0:  # up
            intended = (current_row - 1, current_col)
            perp1, perp2 = (current_row, current_col - 1), (current_row, current_col + 1)  # Left, right
        elif action == 1:  # down
            intended = (current_row + 1, current_col)
            perp1, perp2 = (current_row, current_col - 1), (current_row, current_col + 1)
        elif action == 2:  # left
            intended = (current_row, current_col - 1)
            perp1, perp2 = (current_row - 1, current_col), (current_row + 1, current_col)
        else:  # right
            intended = (current_row, current_col + 1)
            perp1, perp2 = (current_row - 1, current_col), (current_row + 1, current_col)

        # checking if the move can be made , like no obstacle/wall
        def is_valid(pos):
            r, c = pos
            return 0 <= r < self.rows and 0 <= c < self.cols and pos != self.wall

        # trans prob
        next_pos = self.current_pos  # current pos
        reward = -1  # default reward
        done = False

        # getting the next state based on probabilities
        rand = np.random.random()
        if rand < self.p_intended:  # direction
            if is_valid(intended):
                next_pos = intended
            else:
                reward = -5 if intended == self.wall else -1  # penalty on hitting a wall/obstacle
        elif rand < self.p_intended + self.p_perpendicular:
            if is_valid(perp1):
                next_pos = perp1
        else:
            if is_valid(perp2):
                next_pos = perp2

        self.current_pos = next_pos

        # check for the target
        if self.current_pos == self.target:
            reward = 10
            done = True

        state = self.state_to_idx[self.current_pos]
        return state, reward, done, False, {}

    def render(self):
        grid = np.array([[" " for _ in range(self.cols)] for _ in range(self.rows)])
        grid[self.wall] = "W"
        grid[self.target] = "T"
        grid[self.current_pos] = "R"
        print(grid)

### Using Q learning to solve the MDP
> estimating cumulative rewards

In [2]:
import numpy as np

# hyperparameters
num_episodes = 5000
max_steps = 100
alpha = 0.1
gamma = 0.9
epsilon = 0.1

# initialize the environment
env = GridWorldMDP()
num_states = env.observation_space.n
num_actions = env.action_space.n

# initializing the Q-table
Q = np.zeros((num_states, num_actions))

# Q-Learning Algorithm
for episode in range(num_episodes):
    state, _ = env.reset()
    for step in range(max_steps):
        # selection
        if np.random.random() < epsilon:
            action = env.action_space.sample()  # explore
        else:
            action = np.argmax(Q[state, :])  # exploit

        # action
        next_state, reward, done, _, _ = env.step(action)

        # update the Q-value
        Q[state, action] = Q[state, action] + alpha * (
            reward + gamma * np.max(Q[next_state, :]) - Q[state, action]
        )

        state = next_state
        if done:
            break

# testing the learning policy
state, _ = env.reset()
env.render()
done = False
total_reward = 0
steps = 0

while not done and steps < max_steps:
    action = np.argmax(Q[state, :])
    state, reward, done, _, _ = env.step(action)
    total_reward += reward
    steps += 1
    env.render()

print(f"Total reward: {total_reward}, Steps: {steps}")

[['R' ' ' ' ']
 [' ' 'W' ' ']
 [' ' ' ' 'T']]
[[' ' ' ' ' ']
 ['R' 'W' ' ']
 [' ' ' ' 'T']]
[[' ' ' ' ' ']
 [' ' 'W' ' ']
 ['R' ' ' 'T']]
[[' ' ' ' ' ']
 [' ' 'W' ' ']
 [' ' 'R' 'T']]
[[' ' ' ' ' ']
 [' ' 'W' ' ']
 [' ' ' ' 'R']]
Total reward: 7, Steps: 4
